# Part 1 · Notebook 01 — Macro data and point-in-time (ALFRED)

**Sessions:** S2 (growth, inflation, jobs & data releases) · **Lab:** release calendar and macro data for Excel

**You will:**
1. Download key US macro series and turn them into year-over-year rates.
2. Export them to CSV for your Excel release calendar.
3. See how much the **first-published** payrolls numbers differ from today's **revised** numbers, and why a backtest must use what was known **at the time**.

> ✏️ Change only the cells marked **Change me**.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # p1lib.py lives next to this notebook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p1lib as p

p.use_course_style()
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)   # CSV exports for Excel go here
pd.set_option("display.float_format", "{:,.4f}".format)
print("Offline fixtures:" if os.environ.get("P1_FIXTURES") else "Live data from FRED", os.environ.get("P1_FIXTURES", ""))

## 1. Latest values from FRED

✏️ **Change me:** series codes and start date. Codes: `CPIAUCSL` CPI, `PCEPILFE` core PCE, `UNRATE` unemployment rate, `INDPRO` industrial production, `PAYEMS` nonfarm payrolls.

In [ ]:
SERIES = ["CPIAUCSL", "PCEPILFE", "UNRATE", "INDPRO", "PAYEMS"]   # ✏️ Change me
START = "2005-01-01"                                                # ✏️ Change me

raw = pd.concat({code: p.fred_series(code, start=START) for code in SERIES}, axis=1)
raw.tail()

In [ ]:
macro = pd.DataFrame({
    "CPI YoY %": p.yoy_pct(raw["CPIAUCSL"]),
    "Core PCE YoY %": p.yoy_pct(raw["PCEPILFE"]),
    "Unemployment %": raw["UNRATE"],
    "Industrial production YoY %": p.yoy_pct(raw["INDPRO"]),
    "Payrolls monthly change (000s)": raw["PAYEMS"].diff(),
}).dropna(how="all")
macro.tail(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
macro[["CPI YoY %", "Core PCE YoY %"]].plot(ax=axes[0], title="Inflation, year over year (%)")
axes[0].axhline(2, color="#8a8984", lw=1, ls="--"); axes[0].text(axes[0].get_xlim()[0], 2.2, " Fed target 2%", color="#52514e", fontsize=9)
macro["Unemployment %"].plot(ax=axes[1], title="Unemployment rate (%)")
for ax in axes: ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
path = OUT_DIR / "macro_monthly.csv"
macro.to_csv(path)
print("Saved for Excel:", path.resolve())

## 2. First release vs today's value (payrolls)

Payrolls are revised twice in the following months and again once a year. The value in section 1 is **today's revised** value, not what traders saw on release day.

`alfred_vintages` downloads every published version. It needs `FRED_API_KEY`.

In [ ]:
OBS_START = "2019-01-01"      # ✏️ Change me
vint = p.alfred_vintages("PAYEMS", observation_start=OBS_START)
vint.head()

In [ ]:
# Headline number = monthly CHANGE in payrolls, as first published vs as known today
fr = p.first_release(vint)
rows = []
for obs, row in fr.iterrows():
    seen = p.as_of(vint, row["published"])               # the whole series as visible on release day
    prev = obs - pd.DateOffset(months=1)
    if prev in seen.index:
        rows.append({"month": obs, "published": row["published"].date(),
                     "first_change": seen[obs] - seen[prev]})
first = pd.DataFrame(rows).set_index("month")
latest = vint.sort_values("realtime_start").groupby("date")["value"].last()
first["latest_change"] = latest.diff().reindex(first.index)
first["revision"] = first["latest_change"] - first["first_change"]
first.tail(12)

In [ ]:
ax = first[["first_change", "latest_change"]].plot(marker="o", markersize=3,
        title="Payrolls monthly change (thousands): first release vs latest")
ax.axhline(0, color="#8a8984", lw=1); ax.set_xlabel(""); ax.legend(["First release", "Latest (revised)"])
plt.show()
print("Mean absolute revision (thousands):", round(first["revision"].abs().mean(), 1))
first.reindex(first["revision"].abs().sort_values(ascending=False).index).head(5)

## 3. What a backtest would have "known"

`as_of(vintages, date)` returns the series exactly as it looked on that date. Compare what was visible on a release day with today's values for the same months.

In [ ]:
WHEN = "2021-01-15"      # ✏️ Change me: any date after OBS_START
seen_then = p.as_of(vint, WHEN).rename("known on " + WHEN)
compare = pd.concat([seen_then, latest.rename("latest")], axis=1).dropna().tail(6)
compare["difference"] = compare["latest"] - compare.iloc[:, 0]
compare

## 4. Release surprise table (homework)

✏️ Fill in consensus and actual for 3 recent CPI releases (from an economic calendar), then note the S&P 500 and 2-year yield moves in the first 30 minutes.

In [ ]:
surprises = pd.DataFrame([
    # ✏️ Change me: one row per release
    {"release": "CPI m/m", "date": "2025-05-13", "consensus": 0.3, "actual": 0.2, "spx_30m_pct": None, "us2y_30m_bp": None},
])
surprises["surprise"] = surprises["actual"] - surprises["consensus"]
surprises

In [ ]:
p.log_research({"notebook": "01_fred_alfred", "source": "FRED/ALFRED", "series": ",".join(SERIES),
                "note": f"vintages from {OBS_START}; latest values are revised"}).tail(3)

## Questions
1. How large is the average payrolls revision compared with a typical monthly change?
2. Give one example of a trading rule whose backtest would look better with revised data than it would have in real time.
3. Why do markets react to the **surprise** rather than to the level of a release?